<a href="https://colab.research.google.com/github/NoaTal1996/Backbone-Graph/blob/main/src/Topic_Clustring_and_NER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Instalations and Imports

In [124]:
try:
    # Running inside Google Colab → use !pip
    import google.colab
    in_google_colab = True
    print("Running in Colab → installing packages...")
    !pip install -q bertopic
    !pip install -q tweet-preprocessor
    !pip install -q spacy-entity-linker
    !pip install -q emoji
    !pip install sentence-transformers scikit-learn
    !pip install "transformers>=4.40.0" sentencepiece accelerate torch
except ImportError:
    # Not running in Colab
    in_google_colab = False
    pass

In [125]:
import pandas as pd
import pyarrow  # ensures pandas can read/write parquet
import matplotlib.pyplot as plt

import re
import emoji

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

import spacy

from tqdm.auto import tqdm
from IPython.display import display

# Loading and Exporting
[Labeled Datasets for Research on Information Operations](https://zenodo.org/records/14189193)

## Load Data

In [126]:
dataset_name = "Labeled_Datasets/Ecuador"  # folder path
file_name = f"Ecuador_part_10.gzip.parquet"  # file name

# if run local: put data in "../data/<dataset_name>/<file_name>"
# Recommended: working directory is: "Backbone-Graph/src"

In [127]:
from pathlib import Path

# Google Colab
if in_google_colab:
    from google.colab import drive
    drive.mount('/content/drive')
    dataset_path = Path("/content/drive/Shareddrives/KeyBoostAuthor") / dataset_name / file_name
    print("Running inside Colab. Loading from Google Drive...")

# Cluster\Local
else:
    # Recommended: working directory is: "Backbone-Graph/src"
    dataset_path = Path("../data") / dataset_name / file_name
    print(f"Running on local/cluster.\nLoading data from: {dataset_path}")


# Load the parquet file
df = pd.read_parquet(dataset_path)

# for testing
# df = df.sample(n=100, random_state=42).copy()

Running on local/cluster.
Loading data from: ../data/Labeled_Datasets/Ecuador/Ecuador_part_10.gzip.parquet


## Output Path


In [128]:
from datetime import datetime
today = datetime.now().strftime("%Y_%m_%d")


dataset_path = Path(dataset_path)


if in_google_colab:
  output_folder_path = (dataset_path.parents[2] / "Labeled_Dataset_with_Topic_NER" / dataset_path.parent.name / dataset_path.name)
else:
  # working directory is: Backbone-Graph/src
  file_name_no_suffixes = file_name.split(".")[0]
  output_folder_path = Path("../results") / dataset_name / file_name_no_suffixes / today
  # Example: ""../results/Labeled_Datasets/Ecuador/Ecuador_part_4/2025_11_27"



output_folder_path.mkdir(parents=True, exist_ok=True) # Createst the output directory if needed

print(f"output_folder_path = {output_folder_path}")

output_folder_path = ../results/Labeled_Datasets/Ecuador/Ecuador_part_10/2025_12_14


In [129]:
df.head()

,postid,post_text,application_name,post_language,in_reply_to_postid,in_reply_to_accountid,post_time,accountid,account_profile_description,follower_count,following_count,account_creation_date,is_repost,reposted_accountid,reposted_postid,hashtags,urls,account_mentions,is_control
450000,046afb331bf4449c0adde2fc8f57f53da8b7597f2f,"@dc628a347aeea58305ac837f2a8fdb83062355c692 12,698 VOTOS\n\n@a2d7dc6ed59c0274635da3fd008ad3e0408b22cd6e Yo Voto Por #dc628a347aeea58305ac837f2a8fdb83062355c692Trendy \n#Urband5 \n#BTR\n#KCAMexico",62010408238d4a0f958ec97eec3db0a4f69bcd0d11,es,None,dc628a347aeea58305ac837f2a8fdb83062355c692,2017-07-07 23:43:56,e1dc86b29f3be6a246b84dedcdcd6e1dd50bcf7c97,★┋ 𝐒𝐌𝐈𝐋𝐄 𝐀𝐍𝐃 𝐄𝐍𝐉𝐎𝐘 𝐓𝐇𝐄 𝐋𝐈𝐓𝐓𝐋𝐄 𝐓𝐇𝐈𝐍𝐆𝐒 𝐈𝐍 𝐋𝐈𝐅𝐄 ― #̷K̷e̷r̷e̷m̷B̷ü̷r̷s̷i̷n̷ 🤘🏼.≒♡≓.,3368,829,2013-07-22,False,None,None,"[XtiianZechTrendy, Urband5, BTR, KCAMexico]",[],"[dc628a347aeea58305ac837f2a8fdb83062355c692, a2d7dc6ed59c0274635da3fd008ad3e0408b22cd6e]",True
450001,17759e0b3ce3739984653550fd67f10ca35f18cdf7,"@dc628a347aeea58305ac837f2a8fdb83062355c692 12,699 VOTOS\n\n@a2d7dc6ed59c0274635da3fd008ad3e0408b22cd6e Yo Voto Por #dc628a347aeea58305ac837f2a8fdb83062355c692Trendy \n#Urband5 \n#BTR\n#KCAMexico",62010408238d4a0f958ec97eec3db0a4f69bcd0d11,es,None,dc628a347aeea58305ac837f2a8fdb83062355c692,2017-07-07 23:44:09,e1dc86b29f3be6a246b84dedcdcd6e1dd50bcf7c97,★┋ 𝐒𝐌𝐈𝐋𝐄 𝐀𝐍𝐃 𝐄𝐍𝐉𝐎𝐘 𝐓𝐇𝐄 𝐋𝐈𝐓𝐓𝐋𝐄 𝐓𝐇𝐈𝐍𝐆𝐒 𝐈𝐍 𝐋𝐈𝐅𝐄 ― #̷K̷e̷r̷e̷m̷B̷ü̷r̷s̷i̷n̷ 🤘🏼.≒♡≓.,3368,829,2013-07-22,False,None,None,"[XtiianZechTrendy, Urband5, BTR, KCAMexico]",[],"[dc628a347aeea58305ac837f2a8fdb83062355c692, a2d7dc6ed59c0274635da3fd008ad3e0408b22cd6e]",True
450002,f02b1246d853f29cd937a3c2d972ee6eb56c1fe8ed,@2b1dfaaa07be6eb0a41dd1ea164345d9bf57da90f3 Jalapeño \n#MarioBautista \n#NoDigasNada\n#Bautisters \n#KCAMexico,072f1b013621f1b0d9753c98d97f5a0a97a8b68303,es,3418feae1872adbf2b7a82c68487a5585af72f1c6b,2b1dfaaa07be6eb0a41dd1ea164345d9bf57da90f3,2017-07-07 23:44:50,ae1780e78194ebcc74018a36e1d172a12d7d507a84,None,5212,890,2014-09-17,False,None,None,"[MarioBautista, NoDigasNada, Bautisters, KCAMexico]",[],[2b1dfaaa07be6eb0a41dd1ea164345d9bf57da90f3],True
450003,a248f9fb920fb48f4646828656484482a4a9f465e5,@2b1dfaaa07be6eb0a41dd1ea164345d9bf57da90f3 @1374f260f5a52a8ccd9d760e331e4fecd208eece8c \n#MarioBautista \n#NoDigasNada\n#Bautisters \n#KCAMexico,072f1b013621f1b0d9753c98d97f5a0a97a8b68303,und,3418feae1872adbf2b7a82c68487a5585af72f1c6b,2b1dfaaa07be6eb0a41dd1ea164345d9bf57da90f3,2017-07-07 23:45:11,ae1780e78194ebcc74018a36e1d172a12d7d507a84,None,5212,890,2014-09-17,False,None,None,"[MarioBautista, NoDigasNada, Bautisters, KCAMexico]",[],"[2b1dfaaa07be6eb0a41dd1ea164345d9bf57da90f3, 1374f260f5a52a8ccd9d760e331e4fecd208eece8c]",True
450004,6bd8448aa231f9b97fff366b603530578014011613,"#BelindaTrendy #LaloBritoTrendy \n#KCAMexico 1,510",15592fcc82d5064a8f4a3b89774e6cd0bbd2118aa0,und,None,None,2017-07-07 23:45:12,a9cf188a64587fdfe73d1b4a9e0b76ea285680a4ca,Vivir es increíble.,783,718,2012-06-19,False,None,None,"[BelindaTrendy, LaloBritoTrendy, KCAMexico]",[],[],True


# Dataset Statistics

In [130]:
total_posts = len(df)
hashtag_posts = df[df['hashtags'].notna() & (df['hashtags'].str.len() > 0)]
mention_posts = df[df['account_mentions'].notna() & (df['account_mentions'].str.len() > 0)]
url_posts = df[df['urls'].notna() & (df['urls'].str.len() > 0)]

print(f"{'df shape:':<40} {df.shape}")
print(f"{'df posts with hashtags # shape:':<40} {hashtag_posts.shape}  ({len(hashtag_posts)/total_posts*100:.2f}%)")
print(f"{'df posts with account_mentions @ shape:':<40} {mention_posts.shape}  ({len(mention_posts)/total_posts*100:.2f}%)")
print(f"{'df posts with urls shape:':<40} {url_posts.shape}  ({len(url_posts)/total_posts*100:.2f}%)")


print()
print()

print(f"{'**collum name**':<30} **value type**")
for col in df.columns:
    # dtype_str = str(df[col].dtype)  # Convert dtype to string
    # get a sample non-null value if exists
    sample_val = df[col].dropna().iloc[0] if df[col].notna().any() else None
    print(f"{col:<30} | {type(sample_val).__name__}")


print()
print()
print(df['post_language'].value_counts())

df shape:                                (50000, 19)
df posts with hashtags # shape:          (39808, 19)  (79.62%)
df posts with account_mentions @ shape:  (34182, 19)  (68.36%)
df posts with urls shape:                (16675, 19)  (33.35%)


**collum name**                **value type**
postid                         | str
post_text                      | str
application_name               | str
post_language                  | str
in_reply_to_postid             | str
in_reply_to_accountid          | str
post_time                      | Timestamp
accountid                      | str
account_profile_description    | str
follower_count                 | int64
following_count                | int64
account_creation_date          | date
is_repost                      | bool
reposted_accountid             | str
reposted_postid                | str
hashtags                       | ndarray
urls                           | ndarray
account_mentions               | ndarray
is_control          

# Clean Text

In [131]:
def clean_url(url_match):

    url = url_match.group(0)

    # 1. Remove protocol (http:// or https://)
    url = re.sub(r"https?://", "", url)
    url = re.sub(r"www\.", "", url)

    # 2. Remove query parameters (?x=1)
    url = url.split("?")[0]

    # 3. Split into components by "/"
    parts = url.split("/")

    # Domain (first part)
    domain = parts[0].replace(".", "_")  # twitter.com → twitter_com

    # First section (second part of the URL)
    if len(parts) > 1 and parts[1].strip() != "":
        section = parts[1].replace(".", "_")
        token = f" URL_{domain}_{section} "
    else:
        token = f" URL_{domain} "

    return token


In [132]:
def clean_text(text: str) -> str:

    # lowercase
    text = str(text).lower()

    # URLs → domian + first section
    text = re.sub(r"(https?://\S+|www\.\S+)", clean_url, text)

    # Hashtags → HASHTAG_xxx
    text = re.sub(r"#(\w+)", r" HASHTAG_\1 ", text)

    # Mentions → MENTION_xxx
    text = re.sub(r"@(\w+)", r" MENTION_\1 ", text)

    # emoji → EMOJI_xxx
    text = emoji.demojize(text, delimiters=(" EMOJI_", " "))

    # Collapse spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [133]:
df["clean_post_text"] = df["post_text"].apply(clean_text)

## Remove Enttities Enteties

In [134]:
import re
import pandas as pd

URL_RE = re.compile(r"(https?://\S+|www\.\S+)", flags=re.IGNORECASE)
MENTION_RE = re.compile(r"@\w+")
HASHTAG_RE = re.compile(r"#(\w+)")
WHITESPACE_RE = re.compile(r"\s+")

def clean_enteties(text: str) -> str:
    """
    Lightly clean a social-media post while keeping it natural
    for machine translation.

    Operations:
    - Replace URLs with <URL>
    - Replace @username with <USER>
    - Turn #hashtag into 'hashtag' (drop the #)
    - Collapse repeated whitespace
    """
    if not isinstance(text, str):
        text = str(text)

    # lowercase
    text = str(text).lower()

    # 1. Normalize URLs
    text = URL_RE.sub("<URL>", text)

    # 2. Normalize mentions
    text = MENTION_RE.sub("<USER>", text)

    # 3. Normalize hashtags: "#Ecuador2025" -> "Ecuador2025"
    text = HASHTAG_RE.sub(r"\1", text)

    # 4. Collapse whitespace
    text = WHITESPACE_RE.sub(" ", text).strip()

    return text

In [135]:
df["clean_text_no_enteties"] = df["post_text"].astype(str).apply(clean_enteties)

In [136]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 2000)

df[["post_text", "clean_text_no_enteties"]].sample(n=20)

,post_text,clean_text_no_enteties
480218,@832d5d4493ef56a627ed05b2cf84e46c447689fde3 vinte e dois #KCAMexico #BTS,<USER> vinte e dois kcamexico bts
454996,116 votos para Rai 😘 #Raiza #KCAMexico,116 votos para rai 😘 raiza kcamexico
470365,-257\n#LaloBritoTrendy #DannaPaola #SofiaReyes #CaeliTrendy #LlegasteTu #MichaelRonda #IsabellaCastillo #KCAMexico,-257 lalobritotrendy dannapaola sofiareyes caelitrendy llegastetu michaelronda isabellacastillo kcamexico
469363,El mejor día🇲🇽29.04.17\nEste vídeo lo hice hace 3 meses😭 @11b6ed09d3833634f14f977bb3a22578910941f573 \n\n#KarolSevilla #RuggeroPasquarelli #CD9 #KCAMexico #KCAColombia https://7d6181ce7a5051af68b8f0022845939ae2ec0c4686,el mejor día🇲🇽29.04.17 este vídeo lo hice hace 3 meses😭 <USER> karolsevilla ruggeropasquarelli cd9 kcamexico kcacolombia <URL>
459479,17 #DrakeYJosh #KCAMexico,17 drakeyjosh kcamexico
463147,34 #ReggaetonLento #HeyDJ\n#KCAColombia @b169254567d9c625fe6ba55deb92e9c2afb54c6b72 \n@07a06078f0e5dfff17073bc4ac926b7b928c578b62,34 reggaetonlento heydj kcacolombia <USER> <USER>
470045,-236\n#LaloBritoTrendy #DannaPaola #SofiaReyes #CaeliTrendy #LlegasteTu #MichaelRonda #KCAMexico,-236 lalobritotrendy dannapaola sofiareyes caelitrendy llegastetu michaelronda kcamexico
479092,"@563a5ed43b6f110f47bd7fe890946339351942eafc @2baa5e8195d59b97957cde40ed64321dc38d6f057e @fe340f3810b860d11b560aaaeea53ad999348433aa @3cef94b7d7c331e283ad80bb0c032a3002ee580f6a @2b5ca5f7d58841b06da1be2c0401522e39409b5361 @61061c80bc6e9eec3145b9a6609127f8d0c1ec8bc7 63,961 votos para mis héroes @61061c80bc6e9eec3145b9a6609127f8d0c1ec8bc7 .:*･♡\n\n#61061c80bc6e9eec3145b9a6609127f8d0c1ec8bc7 #AlonsoVillalpandoTrendy #Coders #KCAMexico","<USER> <USER> <USER> <USER> <USER> <USER> 63,961 votos para mis héroes <USER> .:*･♡ 61061c80bc6e9eec3145b9a6609127f8d0c1ec8bc7 alonsovillalpandotrendy coders kcamexico"
457579,⚠Recuerden que #BTS está nominado también para los Soribada Awards y aquí está el tutorial de como votar 😄\nCorran y voten #Army💕\n\nJPao🍑 https://b9c5caa1d4632733f3317fe6c4e7e5076f00a4266a,⚠recuerden que bts está nominado también para los soribada awards y aquí está el tutorial de como votar 😄 corran y voten army💕 jpao🍑 <URL>
481149,@81a4eeb2f70b488a6c2af9a29111378cc62e8271b5 #CamilaCabello #HeyMa #KCAMexico \nCC003,<USER> camilacabello heyma kcamexico cc003


# Translation to English

In [137]:
# map post_language codes to NLLB-200 language codes
# https://dl-translate.readthedocs.io/en/latest/available_languages/

def map_to_nllb_code(lang_code: str) -> str:
    """
    Map your dataset's post_language codes to NLLB-200 language codes.
    If not in mup return the same language code.
    Args:
        lang_code (str): Post language code.
    Returns:
        str: NLLB-200 language code.
    """
    nlln_mapping = {
    # Europe
    "en": "eng_Latn",   # English
    "es": "spa_Latn",   # Spanish
    "pt": "por_Latn",   # Portuguese
    "fr": "fra_Latn",   # French
    "de": "deu_Latn",   # German
    "it": "ita_Latn",   # Italian
    "nl": "nld_Latn",   # Dutch
    "ru": "rus_Cyrl",   # Russian
    "pl": "pol_Latn",   # Polish
    "sv": "swe_Latn",   # Swedish
    "ca": "cat_Latn",    # Catalan
    "cy": "cym_Latn",    # Welsh
    "et": "est_Latn",    # Estonian

    # Middle East
    "ar": "arb_Arab",   # Arabic (MSA)
    "he": "heb_Hebr",   # Hebrew
    "tr": "tur_Latn",   # Turkish
    "fa": "pes_Arab",   # Persian/Farsi

    # South Asia
    "hi": "hin_Deva",   # Hindi
    "ur": "urd_Arab",   # Urdu
    "bn": "ben_Beng",   # Bengali

    # East Asia
    "zh": "zho_Hans",   # Chinese Simplified
    "ja": "jpn_Jpan",   # Japanese
    "ko": "kor_Hang",   # Korean

    # Southeast Asia
    "vi": "vie_Latn",   # Vietnamese
    "id": "ind_Latn",   # Indonesian (modern code)
    "in": "ind_Latn",   # Indonesian (legacy ISO code)
    "tl": "tgl_Latn",   # Tagalog (Philippines)

    # Africa
    "sw": "swh_Latn",   # Swahili

    # Americas
    "qu": "quy_Latn",   # Quechua (Ayacucho)
}


    if lang_code in nlln_mapping:
        return nlln_mapping[lang_code]
    else:
        return lang_code

In [138]:
df["post_language"] = df["post_language"].apply(map_to_nllb_code)

In [139]:
# load models

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

MODEL_NAME = "facebook/nllb-200-distilled-600M"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
translation_pipeline = pipeline("translation", model=model, tokenizer=tokenizer, device=device, max_length=1000, batch_size=16)

Using device: cuda


Device set to use cuda


In [140]:
from tqdm.auto import tqdm


def transformers_translator(
    df: pd.DataFrame,
    translation_pipe,
    text_col: str = "clean_text_no_enteties",
    lang_col: str = "post_language",
    out_col: str = "translated_post",
    target_lang: str = "eng_Latn",
    pivot_lang: str = "spa_Latn",
    backtranslate_same_lang: bool = True,
) -> pd.DataFrame:
    """
    Translate df[text_col] → target_lang using NLLB pipeline.

    If src_lang == target_lang and backtranslate_same_lang=True,
    perform back-translation: target_lang → pivot_lang → target_lang.

    Args:
        df (pd.DataFrame):
            Input DataFrame containing text and language labels.
        translation_pipe (transformers.pipeline):
            Pre-loaded translation pipeline (e.g., NLLB-200).
        text_col (str):
            Column containing original text.
        lang_col (str):
            Column containing language codes matching the pipeline
            (e.g., "spa_Latn", "eng_Latn").
        out_col (str):
            Column name for the translated output.
        target_lang (str):
            Target language code (default: "eng_Latn").
        pivot_lang (str):
            Pivot language code for back-translation (default: "spa_Latn").
        backtranslate_same_lang (bool):
            If True, back-translate rows where src_lang == target_lang.

    Returns:
        pd.DataFrame:
            A DataFrame with the same index as df and one column (out_col).
    """

    result_df = pd.DataFrame(index=df.index)
    result_df[out_col] = None

    # Iterate per language with progress bar
    for src_lang, group in tqdm(df.groupby(lang_col), desc="Translating by language"):

        # Build a list[str]
        texts = group[text_col].astype(str).tolist()

        # Keep the original indexes so we can assign back later
        indexes = group.index.to_list()

        translated = []  # empty means "no translation done"

        # Case 1: normal translation src_lang -> target_lang
        if src_lang != target_lang:
            print(f"{len(texts)} posts: {src_lang} → {target_lang}")
            outputs = translation_pipe(
                texts,
                src_lang=src_lang,
                tgt_lang=target_lang,
            )
            translated = [o["translation_text"] for o in outputs]

        # Case 2: back-translation target_lang -> pivot_lang -> target_lang
        if (src_lang == target_lang) and backtranslate_same_lang:
            print(f"{len(texts)} posts: {target_lang} → {pivot_lang} → {target_lang}")
            pivot_outputs = translation_pipe(
                texts,
                src_lang=target_lang,
                tgt_lang=pivot_lang,
            )
            pivot_texts = [o["translation_text"] for o in pivot_outputs]

            back_outputs = translation_pipe(
                pivot_texts,
                src_lang=pivot_lang,
                tgt_lang=target_lang,
            )
            translated = [o["translation_text"] for o in back_outputs]

        if translated:
            result_df.loc[indexes, out_col] = translated

    return result_df

In [ ]:
print(f"df shape: {df.shape}")
translated_df = transformers_translator(df, translation_pipeline)
df = df.drop(columns=["translated_post"], errors="ignore") # Drop the old column if exist
df = df.join(translated_df)

df shape: (50000, 21)


Translating by language:   0%|          | 0/34 [00:00<?, ?it/s]

88 posts: cat_Latn → eng_Latn
16 posts: cs → eng_Latn
15 posts: cym_Latn → eng_Latn
7 posts: da → eng_Latn
152 posts: deu_Latn → eng_Latn
4608 posts: eng_Latn → spa_Latn → eng_Latn


In [ ]:
pd.set_option("display.width", 3000)
pd.set_option("display.max_colwidth", 1500)

df.sample(10)[["clean_text_no_enteties", "translated_post"]]

# Remove Engllish Stopwords

In [ ]:
from nltk.corpus import stopwords
import nltk

In [ ]:
nltk.download('stopwords')

EXTRA_STOPWORDS = [
    'he', 'she', 'they', 'you', 'i', 'we', 'im', 'hes', 'shes', 'its', 'rt', 'amp', 're',
    'theyre', 'dont', 'doesnt', 'would', 'could', 'like', 'know', 'say', 'said', 'one',
    'look', 'looks', 'think', 'oh', 'yeah', 'guy', 'gonna', 'thing', 'someone', 'people',
    'get', 'got', 'thats', 'just', 'really', 'even', 'still', 'want', 'need', 'needs',
    'me', 'to', 'you', 'the', 'that', 'be', 'my', 'get', 'in', 'and',
     'rt', 'the', 'https', 'to', 'my', 'it', 'on', 'you', 'of', 'for', 'https', 'http', 'RT'
]
all_stopwords_english = set(stopwords.words('english') + EXTRA_STOPWORDS)

def remove_stopwords(text):
    return " ".join([w for w in text.lower().split() if w not in all_stopwords_english])


In [ ]:
df["clean_post_text"] = df["clean_post_text"].apply(remove_stopwords)

In [ ]:
df[["post_text", "clean_post_text"]].head(20)

# English Topic Clustering

## Utils


### Posts Topic Stats

In [ ]:
def make_topic_stats(df, topic_col_name):
    """
    Calculate per-topic post counts.

    Args:
        df (pd.DataFrame): Data with topics and 'is_control'.
        topic_col_name (str): Column name of topic labels.

    Returns:
        topic_stats (pd.DataFrame): Colums: [topic, total_count, io_Count, legitimate_posts, io_percent].
    """
    topic_stats = (
        df.groupby(topic_col_name)
          .agg(
              total_count=(topic_col_name, "count"),
              io_count=("is_control", lambda x: (~x).sum())
          )
          .reset_index()
    )
    topic_stats["legitimate_posts"] = topic_stats["total_count"] - topic_stats["io_count"]
    topic_stats["io_percent"] = topic_stats["io_count"] / topic_stats["total_count"] * 100
    return topic_stats

In [ ]:
from matplotlib.lines import Line2D

def plot_topic_stats(topic_stats, topic_col_name, model_name, output_folder_path = ""):
    """
    Plot stacked IO vs legitimate post counts per topic.
    Saves the fig to output_path if provided.

    Args:
        topic_stats (pd.DataFrame): Output of make_topic_stats().
        topic_col_name (str): Topic column to plot.
        output_folder_path (str): Path to *folder* save the plot.

    Returns:
        None
    """
    topics = topic_stats[topic_col_name]
    io_count = topic_stats["io_count"]
    legitimate_count = topic_stats["legitimate_posts"]
    io_percent = topic_stats["io_percent"]

    plt.figure(figsize=(12, 6))

    # 🔴 Bottom: IO posts
    bar_io = plt.bar(topics, io_count, label="IO posts", color="red")

    # 🔵 Top: Non-IO posts stacked above IO
    bar_legit = plt.bar(topics, legitimate_count, bottom=io_count, label="Non-IO posts")

    # Add IO percentage text
    ax = plt.gca()
    for x, io, legit, pct in zip(topics, io_count, legitimate_count, io_percent):
        total_height = io + legit
        ax.text(
            x,
            total_height + 0.5,              # slightly above the bar
            f"{pct:.1f}%",                 # format like 23.5%
            ha="center",
            va="bottom",
            fontsize=8,
            fontweight="regular"
        )

    plt.title(f"Post Count per Topic ({model_name})")
    plt.xlabel("Topic ID")
    plt.ylabel("Number of Posts")
    plt.xticks(topics)
    plt.grid(axis="y")
    
    percent_legend = Line2D([], [], color="none")
    total_posts_legend = Line2D([], [], color="none")
    plt.legend(
        handles=[bar_io, bar_legit, percent_legend, total_posts_legend],
        labels=[
            "IO posts",
            "Non-IO posts",
            "% IO Posts Ratio",
            f"Total posts: {topic_stats['total_count'].sum()}",
        ],
        loc="best",
    )

    
    plt.tight_layout()


    if output_folder_path:
        plot_file_path = output_folder_path / (f"{model_name}_topic_clustering.png")
        plt.savefig(plot_file_path, dpi=300)
        print(f"plot is saved to {plot_file_path}")

    plt.show()

    plt.close()

### Accoutns Topic Stats

In [ ]:
def make_topic_account_stats(
    df: pd.DataFrame,
    topic_col: str,
    author_col: str = "accountid",
    is_io_control: str = "is_control",
    threshold: float = 0.5,
) -> pd.DataFrame:
    """Compute per-topic unique account counts (IO vs control) using an author-level threshold.

    Args:
        df: Post-level DataFrame.
        topic_col: Topic column name.
        author_col: Author identifier column.
        is_io_control: Column where 1 = control, 0 = IO.
        threshold: IO-rate threshold to label an author as IO (1 = IO).

    Returns:
        pd.DataFrame with per-topic counts of unique IO/control accounts and IO percentage.
    """
    # Author-level IO label (1 = IO)
    author_io_tag = ((1 - df.groupby(author_col)[is_io_control].mean()) >= threshold).astype(int)

    # Topic's authors (unique pairs)
    topic_author_df = df[[topic_col, author_col]].drop_duplicates()
    topic_author_df["author_io_tag"] = topic_author_df[author_col].map(author_io_tag)

    topic_account_stats = (
        topic_author_df
        .groupby(topic_col)["author_io_tag"]
        .value_counts()
        .unstack(fill_value=0)
        .rename(columns={0: "control_accounts", 1: "io_accounts"})
        .reset_index()
    )

    topic_account_stats["total_accounts"] = topic_account_stats["control_accounts"] + topic_account_stats["io_accounts"]
    topic_account_stats["io_percent"] = topic_account_stats["io_accounts"] / topic_account_stats["total_accounts"] * 100

    return topic_account_stats

In [ ]:
def plot_topic_account_stats(
    topic_account_stats: pd.DataFrame,
    topic_col_name: str,
    model_name: str,
    output_folder_path: Path | str | None = None,
):
    """Plot IO vs control account counts per topic.

    Args:
        topic_account_stats: Output of make_topic_account_stats().
        topic_col_name: Topic column name.
        model_name: Model name for title / filename.
        output_folder_path: Optional folder to save the plot.

    Returns:
        None
    """
    topics = topic_account_stats[topic_col_name]
    io_count = topic_account_stats["io_accounts"]
    control_count = topic_account_stats["control_accounts"]
    io_percent = topic_account_stats["io_percent"]

    plt.figure(figsize=(12, 6))

    # 🔴 Bottom: IO accounts
    bar_io = plt.bar(topics, io_count, label="IO accounts", color="red")

    # 🔵 Top: Control accounts
    bar_control = plt.bar(
        topics,
        control_count,
        bottom=io_count,
        label="Control accounts",
    )

    # Add IO percentage text
    ax = plt.gca()
    for x, io, ctrl, pct in zip(topics, io_count, control_count, io_percent):
        total = io + ctrl
        ax.text(
            x,
            total + 0.3,
            f"{pct:.1f}%",
            ha="center",
            va="bottom",
            fontsize=8,
            fontweight="regular",
        )

    plt.title(f"Unique Accounts per Topic ({model_name})")
    plt.xlabel("Topic ID")
    plt.ylabel("Number of Unique Accounts")
    plt.xticks(topics)
    plt.grid(axis="y")

    # Legend with extra info lines
    percent_legend = Line2D([], [], color="none")
    total_accounts_legend = Line2D([], [], color="none")

    plt.legend(
        handles=[bar_io, bar_control, percent_legend, total_accounts_legend],
        labels=[
            "IO accounts",
            "Control accounts",
            "% IO Accounts Ratio",
            f"Total accounts: {topic_account_stats["total_accounts"].sum()}",
        ],
        loc="best",
    )

    if output_folder_path:
        output_folder_path = Path(output_folder_path)
        plot_file_path = output_folder_path / f"{model_name}_topic_accounts.png"
        plt.savefig(plot_file_path, dpi=300)
        print(f"Plot saved to {plot_file_path}")

    plt.show()
    plt.close()

### Cleaning and Embeddings

In [ ]:
clean_posts_text_lst = df["clean_post_text"].to_list()

In [ ]:
from sklearn.preprocessing import normalize

# Load model
model = SentenceTransformer("digio/Twitter4SSE")

# Embeddings
embeddings = model.encode(clean_posts_text_lst, show_progress_bar=True)
embeddings = normalize(embeddings)

## BERTopic

In [ ]:
def english_BERTopic_modeling(posts_text_lst, embeddings, output_folder_path= "", min_topic_size=50):
    """
    Run BERTopic using precomputed embeddings.

    Args:
        posts_text_lst (list[str]): English text posts.
        embeddings (np.ndarray): Precomputed embeddings aligned with the posts.
        min_topic_size (int): Minimum cluster size.

    Returns:
        topics, probs, topic_model
    """
    topic_model = BERTopic(
        language="english",
        min_topic_size=min_topic_size,
        verbose=True # show logs
    )

    topics, probs = topic_model.fit_transform(posts_text_lst, embeddings)

    # save model to folder
    if output_folder_path:
        topic_model.save(output_folder_path / "BERTopic_model")
        print(f"BERTopic model is saved to {output_folder_path}")

    return topics, probs, topic_model


In [ ]:
BERTopic_topic, BERT_probs, BERTopic_model = english_BERTopic_modeling(clean_posts_text_lst, embeddings, output_folder_path)

df["BERTopic_topic"] = BERTopic_topic
df["BERTopic_prob"] = BERT_probs


BERTopic_model.get_topic_info()

In [ ]:
def make_topic_posts_table(
    df,
    topic_col="BERTopic_topic",
    prob_col="BERTopic_prob",
    text_col="post_text",
    top_k=5,
    max_chars=300,
):
    top_posts = (
        df[[topic_col, prob_col, text_col]]
        .dropna(subset=[topic_col, prob_col, text_col])
        .sort_values([topic_col, prob_col], ascending=[True, False])
        .groupby(topic_col, as_index=False)
        .head(top_k)
        .copy()
    )

    # rank within each topic (1..top_k)
    top_posts["rank"] = top_posts.groupby(topic_col).cumcount() + 1

    # enforce uniqueness (topic, rank) -> one row
    top_posts = top_posts.drop_duplicates(subset=[topic_col, "rank"], keep="first")

    topic_posts_table = (
        top_posts.pivot(index=topic_col, columns="rank", values=text_col)
        .rename(columns=lambda i: f"post_{i}")
        .reset_index()
    )

    return topic_posts_table

In [ ]:
topic_posts_table = make_topic_posts_table(df, top_k=5)
display(topic_posts_table)

In [ ]:
topic_col_name = "BERTopic_topic"
BERT_topic_stats = make_topic_stats(df, topic_col_name)
plot_topic_stats(BERT_topic_stats, topic_col_name, "BERTopic", output_folder_path)

In [ ]:
topic_col_name = "BERTopic_topic"

topic_account_stats = make_topic_account_stats(df, topic_col_name)
display(topic_account_stats)
plot_topic_account_stats(topic_account_stats, topic_col_name, "BERTopic", output_folder_path)

In [ ]:
try:
    fig = BERTopic_model.visualize_topics(custom_labels=True)
    display(fig)   # show it
except Exception as e:
    print("Not enough topics for visualization")
    print("Error:", e)

## K-Means

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

def auto_kmeans_topics(embeddings, k_min=4, k_max=40, step = 2):
    best_k = None
    best_score = -1
    best_labels = None

    for k in range(k_min, k_max + 1, step):
        print(f"Trying k={k}")
        km = KMeans(n_clusters=k, random_state=42, n_init="auto")
        labels = km.fit_predict(embeddings)
        score = silhouette_score(embeddings, labels)

        if score > best_score:
            best_score = score
            best_k = k
            best_labels = labels

    return best_labels, best_k

In [ ]:
# Add to df
labels, best_k = auto_kmeans_topics(embeddings)
df["K_Means_topic"] = labels
print(f"Best k: {best_k}")
df.head()

In [ ]:
topic_col_name = "K_Means_topic"
topic_stats = make_topic_stats(df, topic_col_name)
plot_topic_stats(topic_stats, topic_col_name, "K-Means", output_folder_path)

In [ ]:
topic_col_name = "K_Means_topic"

topic_account_stats = make_topic_account_stats(df, topic_col_name)
display(topic_account_stats)
plot_topic_account_stats(topic_account_stats, topic_col_name, "K_Means", output_folder_path)

# Named Entity Recognition (NER)

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

# Load model ONCE
tokenizer = AutoTokenizer.from_pretrained("TweebankNLP/bertweet-tb2_wnut17-ner")
ner_model = AutoModelForTokenClassification.from_pretrained("TweebankNLP/bertweet-tb2_wnut17-ner")

ner_pipeline = pipeline(
    "token-classification",
    model=ner_model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)


def extract_ner_entities(posts_lst: list):
    """
    Runs NER in batches on a list of text posts.

    Args:
          posts_lst (list): List of strings (cleaned posts).
    Returns
    -------
    all_entities (list[list[str]]): For each post: a list of extracted entity words.
    """

    ner_outputs = ner_pipeline(posts_lst)
    all_entities = []
    for post_output in ner_outputs:
        words = [ent["word"] for ent in post_output]
        all_entities.append(words)

    return all_entities

In [ ]:
# Apply ONCE, no pandas apply, in order to utilize batches.
df["ner_entities"] = extract_ner_entities(df["clean_post_text"].to_list())

In [ ]:
df.loc[df["ner_entities"].astype(bool), ["clean_post_text", "ner_entities"]].head()

# Entities Cleanup

In [ ]:
def clean_entity_columns(df, entity_cols_lst):
    """Normalize entity columns to match TfidfVectorizer behavior.

    Converts each entity to a stripped, lowercase string for all columns
    listed in ``entity_cols_lst``. 
    The function modifies ``df``.

    Args:
        df: Input DataFrame.
        entity_cols_lst: Columns containing lists of entities.

    Returns:
        None.
    """
    for col in entity_cols_lst:
        df[col] = df[col].apply(lambda lst: [str(e).strip().lower().replace(" ", "_") for e in lst])

In [ ]:
entity_cols_lst = ["hashtags", "account_mentions", "urls", "ner_entities"]
clean_entity_columns(df, entity_cols_lst)

# Entities Statistics

In [ ]:
import pandas as pd

def compute_entity_topic_matrix(df, entity_col, topic_col="topic_id", is_control_col="is_control", output_folder_path=None):
    """
    Create a df where values = number of times the entity appears in each topic.

    Args:
        df (pd.DataFrame):
            DataFrame containing posts, entity lists, topic labels and is_control.
        entity_col (str):
            Column containing a list of entities per post.
        topic_col (str):
            Column containing topic labels.
        is_control_col (str):
            Boolean column: True = legitimate, False = IO.

    Returns:
        entity_topic_counts (pd.DataFrame):
            Columns:
                [entity, total_count, io_percent, topic_ids = 1, 2, 3...]
    """

    # 1) Explode entity lists → one row per (entity, topic, is_control)
    df_exp = (
        df[[entity_col, topic_col, is_control_col]]
        .explode(entity_col)
        .dropna(subset=[entity_col])
        .copy()
    )

    # 2) Aggregate counts per entity (overall)
    entity_stats = (
        df_exp.groupby(entity_col)
              .agg(
                  total_count=(entity_col, "count"),
                  io_count=(is_control_col, lambda x: (~x).sum()),  # False = IO
              )
              .reset_index()
    )
    entity_stats["io_percent"] = 100 * entity_stats["io_count"] / entity_stats["total_count"]
    entity_stats = entity_stats[[entity_col, "total_count", "io_percent"]]

    # 3) Count occurrences per (entity, topic)
    entity_topic_counts = (
        df_exp.groupby([entity_col, topic_col])
              .size()
              .reset_index(name="count")
    )

    # 4) Pivot to wide format (entity rows × topic columns)
    entity_topic_counts = (
        entity_topic_counts
        .pivot_table(
            index=entity_col,
            columns=topic_col,
            values="count",
            fill_value=0
        )
        .sort_index(axis=1)   # ensure topic columns sorted: 0,1,2,...
        .reset_index()
    )

    # 5) Merge total_count + IO stats
    entity_topic_counts = entity_stats.merge(entity_topic_counts, on=entity_col, how="left")

    # 6) Sort by total_count (descending)
    entity_topic_counts = entity_topic_counts.sort_values("total_count", ascending=False)

    # 7) Save if path provided
    if output_folder_path:
        output_path = Path(output_folder_path) / f"entity_topic_matrix.gzip.parquet"
        df.to_parquet(output_path, index=False, compression="gzip")
        print(f"Saved to {output_path}")

    return entity_topic_counts

In [ ]:
import matplotlib.pyplot as plt

def plot_entity_topic_distribution(entity, entity_topic_counts, entity_col, clustering_model="", output_folder_path=None):
    """
    Plot how many times a given entity appears in each topic.
    """

    # Extract row for chosen entity
    row = entity_topic_counts.loc[entity_topic_counts[entity_col] == entity].iloc[0]

    # Keep only topic columns (s)
    topic_cols = [c for c in entity_topic_counts.columns if isinstance(c, int)]
    counts = row[topic_cols]       # Series like [topic_id → count]

    # --- Plot ---
    plt.figure(figsize=(12, 5))
    plt.bar(topic_cols, counts)

    plt.title(f"Topic Distribution for Entity: '{entity}'")
    plt.xlabel("Topic ID")
    plt.ylabel("Count of occurrences")
    plt.xticks(topic_cols)
    plt.grid(axis="y")
    plt.ylim(0, counts.max()* 1.15 ) # y-axis height: 15% padding above the tallest bar
    # Add metadata note (top-right)
    plt.text(
        0.98, 0.92,
        f"clustering model: {clustering_model}\nEntities col: {entity_col}",
        ha="right",
        va="top",
        transform=plt.gca().transAxes
    )

    # plot
    plt.tight_layout()

    # Save if path provided
    if output_folder_path:
        plot_path = output_folder_path / f"{entity}_topic_distribution.png"
        plt.savefig(plot_path, dpi=300)
        print(f"Saved figure to: {plot_path}")

    plt.show()

    plt.close()

In [ ]:
entity_col_name = "ner_entities"
topic_col = "BERTopic_topic"

entity_topic_matrix = compute_entity_topic_matrix(df, entity_col_name, topic_col)
display(entity_topic_matrix.head())

entity = entity_topic_matrix[entity_col_name].iloc[0]   # first entity
plot_entity_topic_distribution(entity=entity, entity_topic_counts=entity_topic_matrix, entity_col=entity_col_name, clustering_model = topic_col, output_folder_path=output_folder_path)

# Save to parquet

In [ ]:
# print dataset statistics
for col in df.columns:
    sample_val = df[col].dropna().iloc[0] if df[col].notna().any() else None
    print(f"{col:<30} | {type(sample_val).__name__}")

In [ ]:
df_output_path = output_folder_path / f"Topic_Ner_{file_name}"

df.to_parquet(df_output_path, index=False, compression="gzip")
print("df is saved to:", df_output_path)